# CPU vs GPU sweep

Goal:

- Run the same sweep, on the same split, on CPU and on GPU.
- Compare hyperparameter sweep time and cost per run.
- Export the best model to `s3://<bucket>/trains/models/`.


## Environment

Install libraries.


In [ ]:
%pip install -q -U ultralytics torch torchvision onnx onnxslim mlflow sagemaker-mlflow

Inspect enironment.


In [ ]:
import json
import os
import sys
import time
from pathlib import Path

import boto3
import matplotlib.pyplot as plt
import mlflow
import pandas as pd
import torch

from sagemaker.core.helper.session_helper import Session

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.tracking import tracking_uri

# define paths
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
RUNS = ROOT / "runs"
MODELS = ROOT / "models"

# create dir
for d in (RAW, PROCESSED, RUNS, MODELS):
    d.mkdir(parents=True, exist_ok=True)

REGION = Session().boto_region_name

# get bucket
env_file = Path.home() / ".sagemaker-yolo.env"
if "BUCKET" not in os.environ and env_file.exists():
    for line in env_file.read_text().splitlines():
        key, _, val = line.partition("=")
        os.environ.setdefault(key.strip(), val.strip())

BUCKET = os.environ["BUCKET"]

# bucket keys
S3_RAW = f"s3://{BUCKET}/data/raw"
S3_SPLIT = f"s3://{BUCKET}/data/split"
S3_MODELS = f"s3://{BUCKET}/trains/models"

# get device info: gpu
HAS_GPU = torch.cuda.is_available()

# set tracking server
TRACKING_URI = tracking_uri()
mlflow.set_tracking_uri(TRACKING_URI)

# set experiment
EXPERIMENT = "yolo-plate-detection-cpu-vs-gpu"
experiment = mlflow.set_experiment(EXPERIMENT)

# get tracking server
server = boto3.client("sagemaker").describe_mlflow_tracking_server(
    TrackingServerName=TRACKING_URI.rsplit("/", 1)[-1]
)
UI_URL = server["TrackingServerUrl"]

# print environment
print("torch     ", torch.__version__)
print("cuda      ", HAS_GPU)
print("gpu       ", torch.cuda.get_device_name(0) if HAS_GPU else "-")
print("cpu count ", os.cpu_count())
print("bucket    ", BUCKET)
print("experiment", EXPERIMENT, f"(id {experiment.experiment_id})")
print(f"\nUI: {UI_URL}/#/experiments/{experiment.experiment_id}")

Identify the instance. The type is recorded on every run, so the comparison can
say which hardware produced which number.

In [ ]:
# Studio writes the space's resource config here
metadata_file = Path("/opt/ml/metadata/resource-metadata.json")
INSTANCE = "unknown"

if metadata_file.exists():
    meta = json.loads(metadata_file.read_text())
    INSTANCE = (
        meta.get("ResourceArn", "").split("/")[-1]
        if "InstanceType" not in meta
        else meta["InstanceType"]
    )
    space = meta.get("SpaceName")
    if space:
        try:
            described = boto3.client("sagemaker").describe_space(
                DomainId=meta["DomainId"], SpaceName=space
            )
            INSTANCE = (
                described["SpaceSettings"]["JupyterLabAppSettings"]
                ["DefaultResourceSpec"]["InstanceType"]
            )
        except Exception as exc:
            print(f"could not read instance type from the space: {exc}")

# passes to run in this session: a GPU box can do both, a CPU box only one
PASSES = [("cpu", "cpu"), ("gpu", 0)] if HAS_GPU else [("cpu", "cpu")]

print("instance  ", INSTANCE)
print("passes    ", [p[0] for p in PASSES])

if not HAS_GPU:
    print(
        "\nNo GPU here, so only the CPU pass runs. To collect the GPU numbers:"
        "\n  1. File > Shut Down all apps (or stop the JupyterLab app in the console)"
        "\n  2. set notebook_instance_type to a GPU type, e.g. ml.g4dn.xlarge"
        "\n  3. terraform apply, restart the app, and run this notebook again"
        "\nThe split is pulled from S3 on the second run, so both passes see"
        "\nexactly the same data."
    )

## Data

Train on the identical split.


In [ ]:
from src.data_loader import build_split, verify_split, write_data_yaml
from src.s3_sync import download, list_objects, upload

# cap the split for a quick check; set to None for the real comparison
LIMIT = 200
# LIMIT = None

SPLIT_SEED = 0

# reuse the split if exists
existing = list_objects(S3_SPLIT)

if existing and LIMIT is None:
    print(f"reusing the split in {S3_SPLIT}/ ({len(existing)} objects)")
    print(download(S3_SPLIT, PROCESSED))
else:
    if existing:
        print(f"LIMIT={LIMIT} set, so rebuilding rather than reusing S3")
    print(download(S3_RAW, RAW))
    print(build_split(RAW, PROCESSED, val_fraction=0.2, limit=LIMIT, seed=SPLIT_SEED))
    print(upload(PROCESSED, S3_SPLIT, delete=True))

print(verify_split(PROCESSED))

names = (RAW / "classes.txt").read_text().split() if (RAW / "classes.txt").exists() else ["car_plate"]
data_yaml = write_data_yaml(ROOT / "configs" / "data.yaml", PROCESSED, names)

n_images = sum(len(list((PROCESSED / s / "images").iterdir())) for s in ("train", "val"))
print(f"\n{n_images} images")

## Define sweep

The same grid runs on both devices.


In [ ]:
from itertools import product

from src.data_loader import build_train_cfg

# the axes to sweep; identical for both devices
SWEEP = {
    "epochs": (10, 20),
}

GRID = [dict(zip(SWEEP.keys(), values)) for values in product(*SWEEP.values())]

print(f"{len(GRID)} configs x {len(PASSES)} passes = {len(GRID) * len(PASSES)} runs")
for g in GRID:
    print(" ", g)

## Run sweep

- One MLflow run per config per device. 
- Run names carry the device.

In [ ]:
from src.tracking import run_sweep

results = []

for tag, device in PASSES:
    base_cfg = build_train_cfg(
        device=device,
        # ultralytics forces 0 dataloader workers on CPU
        workers=(os.cpu_count() or 2) if device != "cpu" else 0,
    )
    base_cfg["project"] = str(ROOT / base_cfg["project"])

    grid = [
        {**g, "name": f"{tag}-{n_images}img-{base_cfg['imgsz']}px-"
                      + "-".join(f"{k}{v}" for k, v in g.items())}
        for g in GRID
    ]

    print(f"\n{'#' * 60}\n# {tag} pass on {INSTANCE}\n{'#' * 60}")

    passed = run_sweep(
        grid=grid,
        base_cfg=base_cfg,
        data_yaml=data_yaml,
        processed_dir=PROCESSED,
        raw_dir=RAW,
        experiment=EXPERIMENT,
        run_name=lambda cfg: cfg["name"],
    )

    # tag each run so the comparison can group by device and hardware
    for r in passed:
        r["pass"] = tag
        r["instance"] = INSTANCE
        if "run_id" in r:
            with mlflow.start_run(run_id=r["run_id"]):
                mlflow.set_tags({
                    "pass": tag,
                    "instance": INSTANCE,
                    "cpu_count": os.cpu_count(),
                    "gpu": torch.cuda.get_device_name(0) if device != "cpu" else "none",
                })

    results.extend(passed)

pd.DataFrame(results)

## Compare

Read both passes back from MLflow.


In [ ]:
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

if runs.empty:
    raise RuntimeError("no runs in the experiment yet")

# elapsed_seconds is logged by run_sweep; tags.pass by the cell above
table = pd.DataFrame({
    "run": runs["tags.mlflow.runName"],
    "pass": runs.get("tags.pass"),
    "instance": runs.get("tags.instance"),
    "epochs": pd.to_numeric(runs.get("params.epochs"), errors="coerce"),
    "images": pd.to_numeric(runs.get("params.data.total_images"), errors="coerce"),
    "seconds": pd.to_numeric(runs.get("metrics.elapsed_seconds"), errors="coerce"),
    "mAP50-95": pd.to_numeric(runs.get("metrics.metrics/mAP50-95B"), errors="coerce"),
}).dropna(subset=["seconds", "pass"])

have = set(table["pass"].unique())
print("passes on the server:", sorted(have))

for label, subset in table.groupby("pass"):
    print(f"  {label:4} {len(subset)} runs on {sorted(set(subset['instance'].dropna()))}")

if {"cpu", "gpu"} - have:
    print(
        f"\nMissing the {sorted({'cpu', 'gpu'} - have)[0]} pass — "
        "the comparison below stays empty until both exist."
    )

table.sort_values(["epochs", "pass"])

### Speedup

Matched on `epochs` and image count, so each row compares like with like.

In [ ]:
BOTH = not ({"cpu", "gpu"} - have)

if not BOTH:
    print("need both passes before this can be computed")
else:
    cpu = table[table["pass"] == "cpu"].set_index(["epochs", "images"])
    gpu = table[table["pass"] == "gpu"].set_index(["epochs", "images"])

    speedup = pd.DataFrame({
        "cpu_min": (cpu["seconds"] / 60).round(1),
        "gpu_min": (gpu["seconds"] / 60).round(1),
        "cpu_mAP": cpu["mAP50-95"].round(4),
        "gpu_mAP": gpu["mAP50-95"].round(4),
    }).dropna()

    speedup["speedup"] = (speedup["cpu_min"] / speedup["gpu_min"]).round(1)
    speedup["mAP_delta"] = (speedup["gpu_mAP"] - speedup["cpu_mAP"]).round(4)

    display(speedup.reset_index())

    print(f"\nmedian speedup {speedup['speedup'].median():.1f}x")
    # different kernels and reduction orders, so exact parity is not expected
    print(f"largest mAP gap {speedup['mAP_delta'].abs().max():.4f}"
          " (small values confirm the two devices agree)")

Wall-clock per config.

In [ ]:
if not BOTH:
    print("need both passes before this can be plotted")
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    pivot = table.pivot_table(index="epochs", columns="pass", values="seconds") / 60
    pivot.plot(kind="bar", ax=axes[0], color={"cpu": "#888", "gpu": "#2a9d8f"})
    axes[0].set_ylabel("minutes")
    axes[0].set_xlabel("epochs")
    axes[0].set_title("wall-clock")
    axes[0].grid(alpha=0.3, axis="y")
    axes[0].tick_params(axis="x", rotation=0)

    acc = table.pivot_table(index="epochs", columns="pass", values="mAP50-95")
    acc.plot(kind="bar", ax=axes[1], color={"cpu": "#888", "gpu": "#2a9d8f"})
    axes[1].set_ylabel("mAP50-95")
    axes[1].set_xlabel("epochs")
    axes[1].set_title("accuracy (should match closely)")
    axes[1].set_ylim(0, 1)
    axes[1].grid(alpha=0.3, axis="y")
    axes[1].tick_params(axis="x", rotation=0)

    plt.tight_layout()
    plt.show()

### Cost


In [ ]:
# USD per hour, on-demand. Update for your region before relying on this.
RATES = {
    "ml.t3.xlarge": 0.20,
    "ml.g4dn.xlarge": 0.74,
    "ml.g5.xlarge": 1.41,
}

if not BOTH:
    print("need both passes before this can be computed")
else:
    cost = table.copy()
    cost["rate"] = cost["instance"].map(RATES)
    unpriced = sorted(set(cost.loc[cost["rate"].isna(), "instance"].dropna()))

    if unpriced:
        print(f"no rate for {unpriced} — add them to RATES")

    cost["usd"] = (cost["seconds"] / 3600 * cost["rate"]).round(3)
    summary = cost.groupby("pass").agg(
        instance=("instance", "first"),
        rate_per_hour=("rate", "first"),
        total_min=("seconds", lambda s: round(s.sum() / 60, 1)),
        total_usd=("usd", "sum"),
    ).round(3)

    display(summary)

    if summary["total_usd"].notna().all() and len(summary) == 2:
        cheaper = summary["total_usd"].idxmin()
        ratio = summary["total_usd"].max() / summary["total_usd"].min()
        print(f"\n{cheaper} is {ratio:.1f}x cheaper for this sweep")